In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/player_match_features.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(df.shape)
print("stumpings" in df.columns, "role" in df.columns)

(27909, 64)
True True


/var/folders/3b/khlc6jhj47qcw7sj4kwtxp5m0000gn/T/ipykernel_10438/365756486.py:4: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/processed/player_match_features.csv")


In [3]:
# ---- Single consolidated build: career stats -> role (incl. wicketkeeper) -> credit_value ----

# 1. Career aggregates, built fresh from df every run
career_stats = df.groupby("player").agg(
    total_matches=("player", "count"),
    career_runs=("runs", "sum"),
    career_balls_faced=("balls_faced", "sum"),
    career_wickets=("wickets", "sum"),
    career_balls_bowled=("balls_bowled", "sum"),
    career_runs_conceded=("runs_conceded", "sum"),
    career_stumpings=("stumpings", "sum"),
    avg_bat_position=("batting_position", lambda x: x.mode()[0])
).reset_index()

career_stats["career_strike_rate"] = np.where(
    career_stats["career_balls_faced"] > 0,
    career_stats["career_runs"] / career_stats["career_balls_faced"] * 100, 0
)
career_stats["career_economy"] = np.where(
    career_stats["career_balls_bowled"] > 0,
    career_stats["career_runs_conceded"] / career_stats["career_balls_bowled"] * 6, 0
)
career_stats["career_batting_avg"] = career_stats["career_runs"] / career_stats["total_matches"]
career_stats["career_wickets_per_match"] = career_stats["career_wickets"] / career_stats["total_matches"]

# 2. Role classification (batter/bowler/allrounder/unknown), then wicketkeeper override
def classify_role(total_matches, career_runs, career_wickets, avg_bat_position, career_balls_bowled):
    is_genuine_bowler = career_wickets >= 20 and career_balls_bowled >= 200
    is_genuine_batter = career_runs >= 500 or (career_runs >= 300 and avg_bat_position <= 7)

    if total_matches > 20:
        if is_genuine_batter and is_genuine_bowler:
            return "allrounder"
        elif is_genuine_bowler:
            return "bowler"
        elif is_genuine_batter:
            return "batter"
        else:
            return "unknown"
    else:
        if avg_bat_position == 0:
            return "bowler"
        elif career_balls_bowled > 50 and avg_bat_position >= 7:
            return "bowler"
        elif avg_bat_position <= 7 and career_balls_bowled > 30:
            return "allrounder"
        elif avg_bat_position <= 7:
            return "batter"
        else:
            return "bowler"

career_stats["role"] = career_stats.apply(
    lambda row: classify_role(row["total_matches"], row["career_runs"], row["career_wickets"],
                               row["avg_bat_position"], row["career_balls_bowled"]),
    axis=1
)

# wicketkeeper override: 3+ career stumpings overrides whatever role was assigned above
career_stats["role"] = career_stats.apply(
    lambda row: "wicketkeeper" if row["career_stumpings"] >= 3 else row["role"],
    axis=1
)

role_map = {"batter": 0, "bowler": 1, "allrounder": 2, "unknown": 3, "wicketkeeper": 4}
career_stats["role_encoded"] = career_stats["role"].map(role_map)

print(career_stats["role"].value_counts())

# 3. Recent form: each player's latest known rolling averages
latest_form = (
    df.sort_values("date").groupby("player").tail(1)
    [["player", "rolling_avg_fantasy_5", "rolling_avg_fantasy_10"]]
    .rename(columns={"rolling_avg_fantasy_5": "recent_form_5", "rolling_avg_fantasy_10": "recent_form_10"})
)
career_stats = career_stats.merge(latest_form, on="player", how="left")

# 4. Role-specific raw score (now including wicketkeeper)
def compute_raw_score(row):
    if row["role"] == "batter":
        return (row["career_batting_avg"] * 1.0) + (row["career_strike_rate"] * 0.15)
    elif row["role"] == "bowler":
        return (row["career_wickets_per_match"] * 25) - (row["career_economy"] * 1.5)
    elif row["role"] == "allrounder":
        return ((row["career_batting_avg"] * 0.6) + (row["career_strike_rate"] * 0.08)
                + (row["career_wickets_per_match"] * 15) - (row["career_economy"] * 1.0))
    elif row["role"] == "wicketkeeper":
        stumpings_per_match = row["career_stumpings"] / row["total_matches"] if row["total_matches"] > 0 else 0
        return (row["career_batting_avg"] * 0.9) + (row["career_strike_rate"] * 0.12) + (stumpings_per_match * 10)
    else:
        return 5.0

career_stats["raw_score"] = career_stats.apply(compute_raw_score, axis=1)

# 5. Normalize career score and recent form, both WITHIN role, with a safe zero-range guard
def safe_normalize(x):
    if x.max() == x.min():
        return pd.Series(0.5, index=x.index)
    return (x - x.min()) / (x.max() - x.min())

career_stats["career_score_norm"] = career_stats.groupby("role")["raw_score"].transform(safe_normalize)

career_stats["recent_form_blend"] = (
    0.6 * career_stats["recent_form_5"].fillna(career_stats["recent_form_10"])
    + 0.4 * career_stats["recent_form_10"].fillna(career_stats["recent_form_5"])
).fillna(0)
career_stats["recent_form_norm"] = career_stats.groupby("role")["recent_form_blend"].transform(safe_normalize)

# 6. Final blend and rescale to a realistic 7.0-10.5 credit range
career_stats["final_score_norm"] = 0.5 * career_stats["career_score_norm"] + 0.5 * career_stats["recent_form_norm"]
career_stats["credit_value"] = (7.0 + career_stats["final_score_norm"] * (10.5 - 7.0)).round(1)

print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya", "MS Dhoni"])]
      [["player", "role", "credit_value"]])

career_stats.to_csv("data/processed/player_credits.csv", index=False)
print("Saved. Shape:", career_stats.shape)

role
bowler          412
batter          267
allrounder       74
wicketkeeper     31
unknown          27
Name: count, dtype: int64
        player          role  credit_value
250  HH Pandya    allrounder           8.5
295  JJ Bumrah        bowler           8.4
443   MS Dhoni  wicketkeeper           8.2
761    V Kohli        batter           9.4
Saved. Shape: (811, 23)


In [4]:
# Sanity check: credit distribution
print(career_stats["credit_value"].describe())
print(career_stats.groupby("role")["credit_value"].mean())

count    811.000000
mean       8.388286
std        0.528659
min        7.000000
25%        8.100000
50%        8.400000
75%        8.700000
max       10.400000
Name: credit_value, dtype: float64
role
allrounder      8.437838
batter          8.246067
bowler          8.440534
unknown         8.629630
wicketkeeper    8.590323
Name: credit_value, dtype: float64


In [5]:
test_2025 = df[df["date"].dt.year >= 2025]
sample_match_id = test_2025["match_id"].iloc[0]
sample_match = df[df["match_id"] == sample_match_id]

print(sample_match_id)
print(sample_match[["player", "team", "opposition", "venue", "date"]].drop_duplicates())

1473438
                player                         team  \
24367        JM Sharma  Royal Challengers Bengaluru   
24368        KH Pandya  Royal Challengers Bengaluru   
24369    A Raghuvanshi        Kolkata Knight Riders   
24370     JR Hazlewood  Royal Challengers Bengaluru   
24371     Rasikh Salam  Royal Challengers Bengaluru   
24372     Harshit Rana        Kolkata Knight Riders   
24373       AD Russell        Kolkata Knight Riders   
24374         VG Arora        Kolkata Knight Riders   
24375        Q de Kock        Kolkata Knight Riders   
24376          V Kohli  Royal Challengers Bengaluru   
24377       D Padikkal  Royal Challengers Bengaluru   
24378         CV Varun        Kolkata Knight Riders   
24379       Yash Dayal  Royal Challengers Bengaluru   
24380          PD Salt  Royal Challengers Bengaluru   
24381   LS Livingstone  Royal Challengers Bengaluru   
24382        AM Rahane        Kolkata Knight Riders   
24383       SH Johnson        Kolkata Knight Riders   
24

In [7]:
import pickle

with open("models/lgbm_final.pkl", "rb") as f: 
    lgbm = pickle.load(f)

lgbm_features = lgbm.feature_name_
print(lgbm_features)


sample_match_features = sample_match[lgbm_features]
lgbm_match_preds = lgbm.predict(sample_match_features)

sample_match = sample_match.copy()
sample_match["lgbm_pred"] = lgbm_match_preds

print(sample_match[["player", "team", "lgbm_pred"]].sort_values("lgbm_pred", ascending=False))



['rolling_avg_fantasy_5', 'rolling_avg_fantasy_10', 'rolling_std_fantasy_10', 'rolling_avg_runs_5', 'rolling_avg_wickets_5', 'batting_position', 'matches_played', 'venue_std_fantasy', 'opposition_std_fantasy', 'venue_first_appearance', 'opposition_first_appearance', 'won_toss', 'expanding_season_fantasy_std', 'is_home', 'role_encoded', 'rolling_bowling_contribution_5', 'rolling_batting_contribution_5', 'venue_avg_innings1', 'venue_avg_innings2', 'venue_avg_total_runs', 'weather_temp', 'weather_humidity', 'weather_dew', 'weather_windspeed', 'weather_precip']
                player                         team  lgbm_pred
24386        SP Narine        Kolkata Knight Riders  56.333468
24380          PD Salt  Royal Challengers Bengaluru  47.622525
24375        Q de Kock        Kolkata Knight Riders  42.658790
24382        AM Rahane        Kolkata Knight Riders  40.740307
24376          V Kohli  Royal Challengers Bengaluru  38.370026
24384          VR Iyer        Kolkata Knight Riders  37.23

In [ ]:
import json

with open("models/ensemble_config.json") as f:
    ensemble_config = json.load(f)

sequence_features = ensemble_config["sequence_features"]
context_features = ensemble_config["context_features"]
SEQ_LEN = ensemble_config["seq_len"]

print(len(sequence_features), len(context_features), SEQ_LEN)

11 21 7


In [9]:
import torch
import torch.nn as nn

class CricketLSTM(nn.Module):
    def __init__(self, seq_features, context_features, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=seq_features, hidden_size=hidden_size,
                             num_layers=2, batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(hidden_size + context_features, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, seq, context):
        lstm_out, (hidden, cell) = self.lstm(seq)
        last_output = lstm_out[:, -1, :]
        combined = torch.cat([last_output, context], dim=1)
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(1)

device = torch.device("mps")
model = CricketLSTM(seq_features=len(sequence_features), context_features=len(context_features), hidden_size=32)
model.load_state_dict(torch.load("models/lstm_final.pt", map_location=device))
model = model.to(device)
model.eval()

def get_player_sequence(player, before_date):
    history = df[(df["player"] == player) & (df["date"] < before_date)].sort_values("date")
    hist_data = history[sequence_features].values
    if len(hist_data) < SEQ_LEN:
        pad_len = SEQ_LEN - len(hist_data)
        padding = np.zeros((pad_len, len(sequence_features)))
        seq = np.vstack([padding, hist_data])
    else:
        seq = hist_data[-SEQ_LEN:]
    return seq

lstm_preds_list = []
for _, row in sample_match.iterrows():
    seq = get_player_sequence(row["player"], row["date"])
    context = row[context_features].values.astype(float)

    seq_t = torch.FloatTensor(seq).unsqueeze(0).to(device)
    context_t = torch.FloatTensor(context).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(seq_t, context_t).cpu().numpy()[0]
    lstm_preds_list.append(pred)

sample_match["lstm_pred"] = lstm_preds_list
sample_match["ensemble_pred"] = 0.5 * sample_match["lgbm_pred"] + 0.5 * sample_match["lstm_pred"]

print(sample_match[["player", "team", "lgbm_pred", "lstm_pred", "ensemble_pred"]].sort_values("ensemble_pred", ascending=False))

                player                         team  lgbm_pred  lstm_pred  \
24386        SP Narine        Kolkata Knight Riders  56.333468  36.357822   
24380          PD Salt  Royal Challengers Bengaluru  47.622525  30.835310   
24376          V Kohli  Royal Challengers Bengaluru  38.370026  40.007744   
24384          VR Iyer        Kolkata Knight Riders  37.233684  38.325279   
24385       RM Patidar  Royal Challengers Bengaluru  32.661119  41.449287   
24375        Q de Kock        Kolkata Knight Riders  42.658790  27.730522   
24382        AM Rahane        Kolkata Knight Riders  40.740307  28.396826   
24381   LS Livingstone  Royal Challengers Bengaluru  37.037850  31.187086   
24377       D Padikkal  Royal Challengers Bengaluru  35.278633  28.047758   
24373       AD Russell        Kolkata Knight Riders  32.534694  26.815203   
24369    A Raghuvanshi        Kolkata Knight Riders  32.837619  22.321735   
24389         RK Singh        Kolkata Knight Riders  25.295505  21.676352   

In [11]:
credits = pd.read_csv("data/processed/player_credits.csv")

match_pool = sample_match.drop(columns=["role", "role_encoded"], errors="ignore").merge(
    credits[["player", "role", "credit_value"]],
    on="player", how="left"
)

print(match_pool[["player", "team", "role", "credit_value", "ensemble_pred"]])

             player                         team          role  credit_value  \
0         JM Sharma  Royal Challengers Bengaluru  wicketkeeper           7.9   
1         KH Pandya  Royal Challengers Bengaluru    allrounder           9.0   
2     A Raghuvanshi        Kolkata Knight Riders        batter           9.3   
3      JR Hazlewood  Royal Challengers Bengaluru        bowler           8.6   
4      Rasikh Salam  Royal Challengers Bengaluru        bowler           8.9   
5      Harshit Rana        Kolkata Knight Riders        bowler           8.6   
6        AD Russell        Kolkata Knight Riders    allrounder           9.4   
7          VG Arora        Kolkata Knight Riders        bowler           8.6   
8         Q de Kock        Kolkata Knight Riders  wicketkeeper           9.3   
9           V Kohli  Royal Challengers Bengaluru        batter           9.4   
10       D Padikkal  Royal Challengers Bengaluru        batter           8.9   
11         CV Varun        Kolkata Knigh

In [12]:
import pulp

players = match_pool["player"].tolist()
points = dict(zip(match_pool["player"], match_pool["ensemble_pred"]))
credits_map = dict(zip(match_pool["player"], match_pool["credit_value"]))
roles = dict(zip(match_pool["player"], match_pool["role"]))
teams = dict(zip(match_pool["player"], match_pool["team"]))

prob = pulp.LpProblem("Fantasy_Team_Selection", pulp.LpMaximize)

player_vars = {p: pulp.LpVariable(f"select_{p}", cat="Binary") for p in players}

# objective: maximize total predicted points
prob += pulp.lpSum([points[p] * player_vars[p] for p in players])

# exactly 11 players
prob += pulp.lpSum([player_vars[p] for p in players]) == 11

# credit budget
prob += pulp.lpSum([credits_map[p] * player_vars[p] for p in players]) <= 100

# role constraints
wk_players = [p for p in players if roles[p] == "wicketkeeper"]
batter_players = [p for p in players if roles[p] == "batter"]
bowler_players = [p for p in players if roles[p] == "bowler"]
allrounder_players = [p for p in players if roles[p] == "allrounder"]

prob += pulp.lpSum([player_vars[p] for p in wk_players]) >= 1
prob += pulp.lpSum([player_vars[p] for p in wk_players]) <= 4

prob += pulp.lpSum([player_vars[p] for p in batter_players]) >= 3
prob += pulp.lpSum([player_vars[p] for p in batter_players]) <= 6

prob += pulp.lpSum([player_vars[p] for p in bowler_players]) >= 3
prob += pulp.lpSum([player_vars[p] for p in bowler_players]) <= 6

prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) >= 1
prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) <= 4

# max 7 players from either team
for team_name in match_pool["team"].unique():
    team_players = [p for p in players if teams[p] == team_name]
    prob += pulp.lpSum([player_vars[p] for p in team_players]) <= 7

# solve
prob.solve()

print("Status:", pulp.LpStatus[prob.status])

selected = [p for p in players if player_vars[p].value() == 1]
result = match_pool[match_pool["player"].isin(selected)].sort_values("ensemble_pred", ascending=False)
print(result[["player", "team", "role", "credit_value", "ensemble_pred"]])
print("Total credits used:", result["credit_value"].sum())
print("Total predicted points:", result["ensemble_pred"].sum())

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/kushpreetsingh/WMK/project101/ipl-fantasy/.venv/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/3b/khlc6jhj47qcw7sj4kwtxp5m0000gn/T/8c678e2626fa4347b35d7b967f2bfee3-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /var/folders/3b/khlc6jhj47qcw7sj4kwtxp5m0000gn/T/8c678e2626fa4347b35d7b967f2bfee3-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 17 COLUMNS
At line 208 RHS
At line 221 BOUNDS
At line 246 ENDATA
Problem MODEL has 12 rows, 24 columns and 118 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 354.562 - 0.00 seconds
Cgl0004I processed model has 8 rows, 24 columns (24 integer (24 of which binary)) and 95 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -354.562
Cbc0038I Before mini bra

In [13]:
print("Status:", pulp.LpStatus[prob.status])
result[["player", "team", "role", "credit_value", "ensemble_pred"]]

Status: Optimal


,player,team,role,credit_value,ensemble_pred
19,SP Narine,Kolkata Knight Riders,allrounder,9.0,46.345645
13,PD Salt,Royal Challengers Bengaluru,batter,9.2,39.228918
9,V Kohli,Royal Challengers Bengaluru,batter,9.4,39.188885
17,VR Iyer,Kolkata Knight Riders,batter,8.8,37.779482
18,RM Patidar,Royal Challengers Bengaluru,batter,9.4,37.055203
8,Q de Kock,Kolkata Knight Riders,wicketkeeper,9.3,35.194656
15,AM Rahane,Kolkata Knight Riders,batter,8.5,34.568566
14,LS Livingstone,Royal Challengers Bengaluru,batter,8.3,34.112468
5,Harshit Rana,Kolkata Knight Riders,bowler,8.6,18.259204
11,CV Varun,Kolkata Knight Riders,bowler,8.7,16.434574


In [14]:
def select_fantasy_team(match_pool, budget=100, min_wk=1, max_wk=4,
                          min_batters=3, max_batters=6,
                          min_bowlers=3, max_bowlers=6,
                          min_allrounders=1, max_allrounders=4,
                          max_per_team=7):
    
    players = match_pool["player"].tolist()
    points = dict(zip(match_pool["player"], match_pool["ensemble_pred"]))
    credits_map = dict(zip(match_pool["player"], match_pool["credit_value"]))
    roles = dict(zip(match_pool["player"], match_pool["role"]))
    teams = dict(zip(match_pool["player"], match_pool["team"]))

    prob = pulp.LpProblem("Fantasy_Team_Selection", pulp.LpMaximize)
    player_vars = {p: pulp.LpVariable(f"select_{p}", cat="Binary") for p in players}

    prob += pulp.lpSum([points[p] * player_vars[p] for p in players])
    prob += pulp.lpSum([player_vars[p] for p in players]) == 11
    prob += pulp.lpSum([credits_map[p] * player_vars[p] for p in players]) <= budget

    wk_players = [p for p in players if roles[p] == "wicketkeeper"]
    batter_players = [p for p in players if roles[p] == "batter"]
    bowler_players = [p for p in players if roles[p] == "bowler"]
    allrounder_players = [p for p in players if roles[p] == "allrounder"]

    prob += pulp.lpSum([player_vars[p] for p in wk_players]) >= min_wk
    prob += pulp.lpSum([player_vars[p] for p in wk_players]) <= max_wk
    prob += pulp.lpSum([player_vars[p] for p in batter_players]) >= min_batters
    prob += pulp.lpSum([player_vars[p] for p in batter_players]) <= max_batters
    prob += pulp.lpSum([player_vars[p] for p in bowler_players]) >= min_bowlers
    prob += pulp.lpSum([player_vars[p] for p in bowler_players]) <= max_bowlers
    prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) >= min_allrounders
    prob += pulp.lpSum([player_vars[p] for p in allrounder_players]) <= max_allrounders

    for team_name in match_pool["team"].unique():
        team_players = [p for p in players if teams[p] == team_name]
        prob += pulp.lpSum([player_vars[p] for p in team_players]) <= max_per_team

    prob.solve(pulp.PULP_CBC_CMD(msg=0))  # msg=0 silences the solver log

    status = pulp.LpStatus[prob.status]
    if status != "Optimal":
        print(f"Warning: solver status is {status}, not Optimal")
        return None

    selected = [p for p in players if player_vars[p].value() == 1]
    result = match_pool[match_pool["player"].isin(selected)].sort_values("ensemble_pred", ascending=False).reset_index(drop=True)

    # captain / vice-captain: top 2 by predicted points among selected 11
    result["captain"] = False
    result["vice_captain"] = False
    result.loc[0, "captain"] = True
    result.loc[1, "vice_captain"] = True

    result["final_points"] = result["ensemble_pred"]
    result.loc[result["captain"], "final_points"] *= 2.0
    result.loc[result["vice_captain"], "final_points"] *= 1.5

    return result

# run it on our test match
team = select_fantasy_team(match_pool)
print(team[["player", "team", "role", "credit_value", "ensemble_pred", "captain", "vice_captain", "final_points"]])
print("\nTotal credits used:", team["credit_value"].sum())
print("Total final fantasy points (with C/VC multipliers):", team["final_points"].sum())

            player                         team          role  credit_value  \
0        SP Narine        Kolkata Knight Riders    allrounder           9.0   
1          PD Salt  Royal Challengers Bengaluru        batter           9.2   
2          V Kohli  Royal Challengers Bengaluru        batter           9.4   
3          VR Iyer        Kolkata Knight Riders        batter           8.8   
4       RM Patidar  Royal Challengers Bengaluru        batter           9.4   
5        Q de Kock        Kolkata Knight Riders  wicketkeeper           9.3   
6        AM Rahane        Kolkata Knight Riders        batter           8.5   
7   LS Livingstone  Royal Challengers Bengaluru        batter           8.3   
8     Harshit Rana        Kolkata Knight Riders        bowler           8.6   
9         CV Varun        Kolkata Knight Riders        bowler           8.7   
10      SH Johnson        Kolkata Knight Riders        bowler           8.1   

    ensemble_pred  captain  vice_captain  final_poi

In [15]:
another_match_id = test_2025[test_2025["match_id"] != sample_match_id]["match_id"].iloc[50]
another_match = df[df["match_id"] == another_match_id].copy()

print(another_match_id)
print(another_match[["player", "team", "opposition", "venue", "date"]].drop_duplicates())

1473441
                player                  team            opposition  \
24439         A Badoni  Lucknow Super Giants        Delhi Capitals   
24440        DA Miller  Lucknow Super Giants        Delhi Capitals   
24441      M Siddharth  Lucknow Super Giants        Delhi Capitals   
24442         N Pooran  Lucknow Super Giants        Delhi Capitals   
24443  J Fraser-McGurk        Delhi Capitals  Lucknow Super Giants   
24444          V Nigam        Delhi Capitals  Lucknow Super Giants   
24445          RR Pant  Lucknow Super Giants        Delhi Capitals   
24446    Abishek Porel        Delhi Capitals  Lucknow Super Giants   
24447     Mukesh Kumar        Delhi Capitals  Lucknow Super Giants   
24448         T Stubbs        Delhi Capitals  Lucknow Super Giants   
24449         AR Patel        Delhi Capitals  Lucknow Super Giants   
24450  Ashutosh Sharma        Delhi Capitals  Lucknow Super Giants   
24451       AK Markram  Lucknow Super Giants        Delhi Capitals   
24452       

In [16]:
def predict_and_select_team(match_df, lgbm_model, lstm_model, lgbm_features,
                              sequence_features, context_features, seq_len,
                              full_df, credits_df, device):
    
    match_df = match_df.copy()
    
    # LightGBM predictions
    lgbm_preds = lgbm_model.predict(match_df[lgbm_features])
    match_df["lgbm_pred"] = lgbm_preds
    
    # LSTM predictions
    lstm_preds_list = []
    for _, row in match_df.iterrows():
        history = full_df[(full_df["player"] == row["player"]) & (full_df["date"] < row["date"])].sort_values("date")
        hist_data = history[sequence_features].values
        if len(hist_data) < seq_len:
            pad_len = seq_len - len(hist_data)
            padding = np.zeros((pad_len, len(sequence_features)))
            seq = np.vstack([padding, hist_data])
        else:
            seq = hist_data[-seq_len:]
        
        context = row[context_features].values.astype(float)
        seq_t = torch.FloatTensor(seq).unsqueeze(0).to(device)
        context_t = torch.FloatTensor(context).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred = lstm_model(seq_t, context_t).cpu().numpy()[0]
        lstm_preds_list.append(pred)
    
    match_df["lstm_pred"] = lstm_preds_list
    match_df["ensemble_pred"] = 0.5 * match_df["lgbm_pred"] + 0.5 * match_df["lstm_pred"]
    
    # merge credits and role
    match_pool = match_df.drop(columns=["role", "role_encoded"], errors="ignore").merge(
        credits_df[["player", "role", "credit_value"]], on="player", how="left"
    )
    
    # optimize
    team = select_fantasy_team(match_pool)
    return team, match_pool

# run on the new match
another_team, another_pool = predict_and_select_team(
    another_match, lgbm, model, lgbm_features,
    sequence_features, context_features, SEQ_LEN,
    df, credits, device
)

print(another_team[["player", "team", "role", "credit_value", "ensemble_pred", "captain", "vice_captain", "final_points"]])
print("\nTotal credits used:", another_team["credit_value"].sum())
print("Total final fantasy points:", another_team["final_points"].sum())

             player                  team          role  credit_value  \
0         DA Miller  Lucknow Super Giants        batter           8.5   
1   J Fraser-McGurk        Delhi Capitals        batter           8.7   
2      F du Plessis        Delhi Capitals        batter           8.7   
3          AR Patel        Delhi Capitals    allrounder           8.9   
4        AK Markram  Lucknow Super Giants        batter           8.5   
5          MR Marsh  Lucknow Super Giants    allrounder          10.2   
6           RR Pant  Lucknow Super Giants  wicketkeeper           9.0   
7          N Pooran  Lucknow Super Giants  wicketkeeper           9.1   
8     Kuldeep Yadav        Delhi Capitals        bowler           8.3   
9          MA Starc        Delhi Capitals        bowler           8.9   
10          V Nigam        Delhi Capitals        bowler           8.5   

    ensemble_pred  captain  vice_captain  final_points  
0       35.368084     True         False     70.736168  
1       3